# Hotel Training - Google Hotels Search 2026-06-01

Notebook ini khusus untuk pipeline hotel, terpisah dari `wisata_training.ipynb`.

Tujuan:
- Membaca JSON Google Hotels Search.
- Mengubah hasil per halaman menjadi CSV raw.
- Membuat CSV training yang sudah deduplicate berdasarkan `property_token`.
- Menambahkan segment properti, quality flags, rating-based sentiment, dan review confidence.

Hasil terakhir:
- Halaman: 29
- Raw records: 511
- Unique training records: 446
- Raw CSV: `D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\01_Raw_Data\generated_raw_csv\HOTEL_GOOGLE_SEARCH_RAW_2026-06-01.csv`
- Training CSV: `D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\HOTEL_TRAINING_GOOGLE_SEARCH_2026-06-01.csv`


In [ ]:
# Rebuild hotel training dataset dari JSON sumber.
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "Penginapan_Workspace" / "06_Scripts" / "build_hotel_training_from_google_hotels_json.py")],
    check=True,
)


In [ ]:
# Baca ringkasan hasil pipeline.
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

summary_path = PROJECT_ROOT / "Penginapan_Workspace" / "02_Curated" / "hotel_training_google_search_summary_2026-06-01.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary


In [ ]:
# Preview data training hotel.
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

training_csv = PROJECT_ROOT / "Penginapan_Workspace" / "02_Curated" / "HOTEL_TRAINING_GOOGLE_SEARCH_2026-06-01.csv"
hotel_training_df = pd.read_csv(training_csv)

hotel_training_df[
    [
        "hotel_training_id",
        "name",
        "property_segment",
        "overall_rating",
        "reviews",
        "price_lowest",
        "adjusted_rating_sentiment_score",
        "review_confidence_label",
        "data_quality_score",
    ]
].head(10)


## Aturan penggunaan

- Gunakan CSV training untuk recommender, bukan CSV raw, karena raw masih berisi duplikat antar halaman.
- Untuk klaim harga, wajib `price_available == True`.
- Untuk klaim fasilitas, wajib `amenities_available == True`.
- Untuk kartu visual, wajib `image_available == True`.
- Untuk ranking berbasis rating, prioritaskan `review_confidence_label` medium atau high.
- Karena sumber ini memakai `vacation_rentals=True`, jangan anggap semua baris sebagai hotel formal.


## Fase A - JSON Google Hotels Search ke CSV Raw Master

Bagian ini mendokumentasikan proses mengubah seluruh JSON Google Hotels Search di workspace penginapan menjadi CSV tabular.

Tujuan tahap ini:

- Membaca semua file JSON di `Penginapan_Workspace/01_Raw_Data/google_hotels_search_json/`.
- Mengabaikan file JSON yang duplikat penuh berdasarkan SHA256.
- Membuka struktur `properties` dan `ads`.
- Membuat CSV raw master yang masih berisi data mentah tabular.
- Membuat inventory sumber agar query, jumlah page, jumlah record, dan duplicate file bisa diaudit.

Hasil terakhir:

- JSON terdeteksi: 18
- JSON diproses: 17
- Duplicate file: 1
- Raw records: 2840
- Unique names raw: 1377
- Raw master CSV: `D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\01_Raw_Data\generated_raw_csv\HOTEL_GOOGLE_SEARCH_RAW_MASTER_2026-06-02.csv`
- Source inventory CSV: `D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\01_Raw_Data\generated_raw_csv\HOTEL_GOOGLE_SEARCH_JSON_SOURCE_INVENTORY_2026-06-02.csv`

Catatan: tahap ini belum dedupe properti dan belum membangun canonical dataset.


In [ ]:
# Rebuild CSV raw master dari seluruh JSON Google Hotels Search.
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "Penginapan_Workspace" / "06_Scripts" / "flatten_google_hotels_json_to_raw_master.py"),
        "--skip-notebook-update",
    ],
    check=True,
)


In [ ]:
# Baca ringkasan dan inventory sumber JSON.
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

summary_path = PROJECT_ROOT / "Penginapan_Workspace" / "02_Curated" / "hotel_google_search_raw_master_summary_2026-06-02.json"
inventory_csv = PROJECT_ROOT / "Penginapan_Workspace" / "01_Raw_Data" / "generated_raw_csv" / "HOTEL_GOOGLE_SEARCH_JSON_SOURCE_INVENTORY_2026-06-02.csv"

summary = json.loads(summary_path.read_text(encoding="utf-8"))
inventory_df = pd.read_csv(inventory_csv)

summary, inventory_df[[
    "source_file",
    "duplicate_status",
    "processed",
    "query",
    "page_count_local",
    "records_total",
    "unique_names_raw",
]].sort_values(["processed", "source_file"], ascending=[False, True]).head(25)


In [ ]:
# Preview CSV raw master.
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

raw_master_csv = PROJECT_ROOT / "Penginapan_Workspace" / "01_Raw_Data" / "generated_raw_csv" / "HOTEL_GOOGLE_SEARCH_RAW_MASTER_2026-06-02.csv"
raw_master_df = pd.read_csv(raw_master_csv)

raw_master_df[[
    "raw_master_id",
    "source_file",
    "source_bucket",
    "source_query_area",
    "name",
    "raw_type",
    "property_segment",
    "latitude",
    "longitude",
    "overall_rating",
    "reviews",
    "price_lowest",
    "data_quality_score",
]].head(20)


### Interpretasi Tahap Ini

CSV raw master adalah hasil tabular dari JSON, bukan dataset final.

Yang sudah aman pada tahap ini:

- JSON sudah berubah menjadi tabel.
- Source lineage per baris tersedia melalui `source_file`, `source_file_sha256`, `source_page_number`, dan `source_query_area`.
- File duplikat penuh tidak ikut diproses ke raw master.
- Kolom dasar seperti koordinat, rating, harga, fasilitas, gambar, dan quality flag awal sudah tersedia.

Langkah berikutnya setelah tahap ini:

1. Dedupe antar properti.
2. Region validation Bandung Raya.
3. Property type classification lanjutan.
4. Build `PENGINAPAN_CANONICAL_CANDIDATE_BANDUNG_RAYA`.


## Fase A - Dedupe dan Region Validation Penginapan

Bagian ini membuat `PENGINAPAN_CANONICAL_CANDIDATE_BANDUNG_RAYA_2026-06-02.csv`.

Keputusan penting:

- Nama output memakai `PENGINAPAN`, bukan `HOTEL`, karena data mencakup hotel, villa, apartment, guest house, homestay, vacation rental, dan room-level listing.
- Dedupe dibuat konservatif agar tidak salah menggabungkan properti berbeda.
- Region validation masih berbasis koordinat rough Bandung Raya, bukan batas administratif presisi.

Hasil terakhir:

- Pool rows: 3021
- Candidate rows: 1656
- Dedupe removed rows: 1365
- Validation gate: PASS_WITH_WARNINGS
- Candidate CSV: `D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_CANONICAL_CANDIDATE_BANDUNG_RAYA_2026-06-02.csv`


In [ ]:
# Rebuild dedupe + region validation penginapan.
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "Penginapan_Workspace" / "06_Scripts" / "build_penginapan_canonical_candidate.py"),
        "--skip-notebook-update",
    ],
    check=True,
)


In [ ]:
# Preview hasil canonical candidate penginapan.
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

candidate_csv = PROJECT_ROOT / "Penginapan_Workspace" / "02_Curated" / "PENGINAPAN_CANONICAL_CANDIDATE_BANDUNG_RAYA_2026-06-02.csv"
summary_json = PROJECT_ROOT / "Penginapan_Workspace" / "02_Curated" / "penginapan_canonical_candidate_summary_2026-06-02.json"
validation_json = PROJECT_ROOT / "Penginapan_Workspace" / "02_Curated" / "penginapan_canonical_candidate_validation_2026-06-02.json"

candidate_df = pd.read_csv(candidate_csv)
summary = json.loads(summary_json.read_text(encoding="utf-8"))
validation = json.loads(validation_json.read_text(encoding="utf-8"))

summary, validation, candidate_df[[
    "penginapan_id",
    "name",
    "property_type",
    "region_bucket",
    "region_validation_label",
    "overall_rating",
    "reviews",
    "data_quality_score",
    "dedupe_group_size",
]].head(20)


## Fase A - Automated Audit Penginapan Canonical Candidate

Bagian ini mendokumentasikan audit otomatis untuk `PENGINAPAN_CANONICAL_CANDIDATE_BANDUNG_RAYA_2026-06-02.csv`.

Audit ini tidak mengubah data. Tujuannya adalah memberi gate dan daftar penyesuaian:

- Schema dan kolom wajib.
- ID unik.
- Koordinat dan region validation.
- Rating, reviews, harga, dan quality score.
- Property type.
- Room-level listing.
- Duplikasi kuat dan kemungkinan duplikasi.
- Missing field penting yang perlu caveat.

Hasil terakhir:

- Gate: `PASS_WITH_WARNINGS`
- Rows: 1656
- Findings: 2945
- Affected rows: 1274
- Findings CSV: `D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_CANONICAL_CANDIDATE_AUTOMATED_AUDIT_FINDINGS_2026-06-02.csv`
- Manual review queue: `D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_MANUAL_REVIEW_QUEUE_AUTOMATED_AUDIT_2026-06-02.csv`
- Adjustment priority queue: `D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_AUTOMATED_AUDIT_ADJUSTMENT_PRIORITY_2026-06-02.csv`


In [ ]:
# Jalankan ulang automated audit candidate penginapan.
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "Penginapan_Workspace" / "06_Scripts" / "audit_penginapan_canonical_candidate.py"),
        "--skip-notebook-update",
    ],
    check=True,
)


In [ ]:
# Baca hasil audit otomatis.
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

summary_path = PROJECT_ROOT / "Penginapan_Workspace" / "02_Curated" / "penginapan_canonical_candidate_automated_audit_2026-06-02.json"
findings_csv = PROJECT_ROOT / "Penginapan_Workspace" / "02_Curated" / "PENGINAPAN_CANONICAL_CANDIDATE_AUTOMATED_AUDIT_FINDINGS_2026-06-02.csv"
manual_queue_csv = PROJECT_ROOT / "Penginapan_Workspace" / "02_Curated" / "PENGINAPAN_MANUAL_REVIEW_QUEUE_AUTOMATED_AUDIT_2026-06-02.csv"
adjustment_queue_csv = PROJECT_ROOT / "Penginapan_Workspace" / "02_Curated" / "PENGINAPAN_AUTOMATED_AUDIT_ADJUSTMENT_PRIORITY_2026-06-02.csv"

audit_summary = json.loads(summary_path.read_text(encoding="utf-8"))
findings_df = pd.read_csv(findings_csv)
manual_queue_df = pd.read_csv(manual_queue_csv)
adjustment_queue_df = pd.read_csv(adjustment_queue_csv)

audit_summary, findings_df["severity"].value_counts().to_dict(), findings_df["code"].value_counts().head(15).to_dict(), adjustment_queue_df.head(30)
